# Stage 2 · Frontier LLMs on the same tests

**Tiny Aya SMS Extract** · Anshul J Amin · Lexsi AI Safety & Alignment Hackathon, Track 1

Can a big general model do this task without fine-tuning? We run **Cohere Command A** (closed, Cohere API) and **Llama 3.3 70B Instruct** (open, Hugging Face Inference Providers) on exactly the same messages as our fine-tuned Tiny Aya: seen formats, unseen formats, hand-written, their rephrased versions and the unrelated texts.

Fair setup: the same system prompt with every schema rule, the same three few-shot examples Tiny Aya's few-shot baseline used, JSON mode on, temperature 0. Scoring: the same scorer and **AuditKIT**. We also measure tokens (for cost per 1M SMS) and latency, and compare with Tiny Aya on the same message ids (McNemar).

**How to run (Kaggle):** *File → Import Notebook* → this file. **Internet ON**, no accelerator. Secrets: `COHERE_API_KEY`, `HF_TOKEN`. Run `QUICK_MODE = True` first, then `QUICK_MODE = False`.

In [ ]:
import os, sys, json, time, random, subprocess, threading, re
import concurrent.futures as cf
from collections import Counter

QUICK_MODE = False
SEED = 7
V3_DATA_REPO = "Meowniece/tiny-aya-sms-extract-v3-data"
TINY_AYA_RESULTS_REPO = "Meowniece/tiny-aya-sms-extract-fire-v3-data"   # written by stage2_train_eval.ipynb

SPLITS = ["test_id", "test_ood", "challenge", "test_id_rephrased", "test_ood_rephrased", "challenge_rephrased", "unrelated"]
SETTINGS = ["few_shot"]          # add "zero_shot" if your budget allows (doubles the calls)

RUN_COMMAND = True
COMMAND_PREFERENCE = ["command-a-03-2025"]      # auto-detected from your Cohere account below
COMMAND_RPM = 18                                 # trial keys: 20 calls/min and ~1,000 calls/month (not enough for all
                                                 # 1,700 messages: use a production key, or limit SPLITS)
RUN_OPEN = True
OPEN_MODEL = "meta-llama/Llama-3.3-70B-Instruct"  # Hugging Face Inference Providers
OPEN_RPM = 60
HF_BILL_TO = ""                                   # e.g. the hackathon org name, to bill its credits; "" = your account

# USD per 1M tokens (input, output). Check the current list prices before reporting.
PRICES = {"command_a": (2.50, 10.00), "llama_3_3_70b": (0.88, 0.88)}

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work_frontier")
for d in ["src", "data", "results", "cache"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
N_PER_SPLIT = 6 if QUICK_MODE else None
print("work dir:", WORK, "| quick mode:", QUICK_MODE)

## 1. Install and read secrets

In [ ]:
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "FAIL ") + " ".join(args))
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0
OK_CURATOR = pip("curatorkit")
OK_AUDIT = pip("auditkit")
pip("litellm", "huggingface_hub", "requests")

def secret(name):
    if os.environ.get(name): return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
for k in ["COHERE_API_KEY", "HF_TOKEN"]:
    v = secret(k)
    if v: os.environ[k] = v
    print(k, "found" if v else "MISSING")

## 2. Project source files

In [ ]:
# ---- src/smsgen.py ----
CONTENT = '"""Synthetic Indian bank / UPI SMS generator with exact gold labels.\n\nDesign\n------\n* Every template is a list of SEGMENTS joined by \'‖\'. The first segment is the core\n  (always kept); later segments can be dropped (optional \'?\' or by truncation). A field\n  is in the gold label ONLY if its slot survived -> the gold never contains information\n  that is not in the text (this is what lets us measure hallucination honestly).\n* \'{@inst}\' reveals the instrument without printing it (e.g. "UPI Ref" text implies UPI).\n* The LAST template of each (family, language) is held out -> `test_ood` uses message\n  formats never seen in training.\n* Bank names are FICTIONAL and messages contain no links or phone numbers, so the data\n  cannot be confused with, or reused as, real bank messages.\n"""\nfrom __future__ import annotations\n\nimport json\nimport random\nimport re\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nFIELDS = ["is_transaction", "type", "amount", "currency", "account_last4", "instrument",\n          "counterparty", "date", "reference", "balance", "status"]\nINSTRUMENTS = ["UPI", "CARD", "NEFT", "IMPS", "RTGS", "ATM", "NACH", "AUTOPAY"]\nLANGS = ["en", "hi", "hinglish"]\n\nFICTIONAL_BANKS = ["Kestrel Bank", "Banyan Bank", "Peacock Bank", "Monsoon Bank",\n                   "Neem Bank", "Saffron Bank", "Lotus Demo Bank", "Indigo Coop Bank"]\nVPA_HANDLES = ["@kestrel", "@banyan", "@peacockpay", "@monsoon", "@neemupi", "@saffron"]\n\nMERCHANTS = ["SWIGGY", "ZOMATO", "AMAZON", "FLIPKART", "BIGBASKET", "IRCTC", "UBER", "OLA",\n             "MYNTRA", "DMART", "RELIANCE FRESH", "APOLLO PHARMACY", "BOOKMYSHOW",\n             "SHARMA GENERAL STORE", "GUPTA SWEETS", "CAFE COFFEE CORNER", "METRO CASH CARRY",\n             "HP PETROL PUMP", "DECATHLON", "NYKAA", "CROMA", "BLINKIT", "ZEPTO"]\nPERSONS = ["RAMESH KUMAR", "PRIYA SHARMA", "ANIL VERMA", "SUNITA DEVI", "ARJUN NAIR",\n           "FATIMA SHAIKH", "GURPREET SINGH", "MEERA IYER", "ROHIT DAS", "KAVYA REDDY",\n           "IMRAN KHAN", "POOJA PATEL", "VIKRAM JOSHI", "ANJALI MISHRA", "SURESH YADAV"]\nEMPLOYERS = ["ACME TECHNOLOGIES PVT LTD", "SUNRISE SOFTWARE LLP", "GANGA TEXTILES LTD",\n             "NORTHSTAR LOGISTICS", "BLUEWAVE SERVICES PVT LTD", "VEDA HOSPITALS"]\nSUBSCRIPTIONS = ["NETFLIX", "SPOTIFY", "YOUTUBE PREMIUM", "JIO FIBER", "AIRTEL POSTPAID",\n                 "TATA PLAY", "GOOGLE ONE", "HOTSTAR", "LIC PREMIUM", "SIP MUTUAL FUND"]\nBILLERS = ["CITY POWER LTD", "JAL BOARD", "METRO GAS", "AIRTEL POSTPAID", "JIO FIBER",\n           "MUNICIPAL TAX"]\nFOREIGN = [("USD", "$"), ("EUR", "€"), ("GBP", "£"), ("AED", "AED ")]\nMONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]\n\n# ---------------------------------------------------------------------------\n# Templates: family -> {"meta": {...}, "en": [...], "hi": [...], "hinglish": [...]}\n# meta: is_txn, type, status (per-template override via tuple (text, status)),\n#       inst (choices for {inst}/{@inst}), cp (counterparty pool name)\n# ---------------------------------------------------------------------------\nT = {\n    "upi_debit": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["UPI"], cp="upi_party"),\n        "en": [\n            "Your A/c {acct} is debited for {amt}‖ on {date}‖ towards {cp}‖ via {inst}‖. UPI Ref No {ref}{@inst}‖. Avl Bal {bal}",\n            "{bank}: {amt} debited from a/c {acct}‖ on {date}‖ to VPA {cp}{@inst}‖ (UPI Ref {ref}{@inst})‖. Avl bal {bal}",\n            "Sent {amt} from {bank} A/c {acct} to {cp} on {date}‖. UPI Ref: {ref}{@inst}‖. Bal: {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} से {amt} डेबिट किए गए‖ दिनांक {date}‖, {cp} को‖ {inst} द्वारा‖। UPI संदर्भ {ref}{@inst}‖। उपलब्ध शेष {bal}",\n            "{bank}: खाता {acct} से {amt} {cp} को UPI से भेजे गए{@inst}‖ ({date})‖। संदर्भ संख्या {ref}‖। शेष राशि {bal}",\n            "{amt} का UPI भुगतान{@inst} {cp} को सफल रहा, खाता {acct}‖, दिनांक {date}‖। रेफ़ {ref}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} se {amt} debit hue‖ {date} ko‖ {cp} ko‖ {inst} se‖. UPI Ref {ref}{@inst}‖. Avl bal {bal}",\n            "{bank}: A/c {acct} se {amt} {cp} ko UPI se bheje gaye{@inst}‖ ({date})‖. Ref no {ref}‖. Bal {bal}",\n            "{amt} ka UPI payment{@inst} {cp} ko successful, A/c {acct}‖, date {date}‖. Ref {ref}",\n        ],\n    },\n    "upi_credit": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=["UPI"], cp="upi_party"),\n        "en": [\n            "Dear Customer, your A/c {acct} is credited with {amt}‖ on {date}‖ by {cp}‖ via {inst}‖. Ref {ref}‖. Avl Bal {bal}",\n            "{amt} received in {bank} a/c {acct} from {cp}‖ on {date}‖. UPI Ref:{ref}{@inst}",\n            "Money received! {cp} sent you {amt} via {inst}‖ on {date}‖. Credited to A/c {acct}‖. Bal {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} में {amt} जमा किए गए‖ दिनांक {date}‖, {cp} द्वारा‖ {inst} से‖। संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} ने आपको {amt} भेजे‖, खाता {acct} में जमा‖ ({date})‖। UPI रेफ़ {ref}{@inst}",\n            "खाता {acct} में {cp} से {amt} प्राप्त हुए‖ {date} को‖ {inst} द्वारा‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} mein {amt} credit hue‖ {date} ko‖ {cp} se‖ {inst} dwara‖. Ref {ref}‖. Avl bal {bal}",\n            "{cp} ne aapko {amt} bheje‖, A/c {acct} mein jama‖ ({date})‖. UPI Ref {ref}{@inst}",\n            "A/c {acct} mein {cp} se {amt} mile‖ {date} ko‖ {inst} se‖. Bal {bal}",\n        ],\n    },\n    "card_spend": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["CARD"], cp="merchant", foreign=0.25),\n        "en": [\n            "You have spent {amt} on your {bank} Credit Card {card}‖ at {cp}‖ on {date}‖. Avl limit: {limit}",\n            "Txn of {amt} on Card {card} at {cp}‖ on {date}‖. Ref {ref}‖. Available credit limit {limit}",\n            "{bank} Debit Card {card} used for {amt} at {cp}‖ on {date} {time}‖. Avl bal {bal}",\n        ],\n        "hi": [\n            "आपके {bank} क्रेडिट कार्ड {card} से {cp} पर {amt} खर्च किए गए‖ दिनांक {date}‖। उपलब्ध लिमिट {limit}",\n            "कार्ड {card} से {amt} का लेनदेन {cp} पर‖, {date}‖। संदर्भ {ref}‖। उपलब्ध क्रेडिट लिमिट {limit}",\n            "{bank} डेबिट कार्ड {card} का उपयोग {cp} पर {amt} के लिए किया गया‖, {date} {time}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke {bank} Credit Card {card} se {cp} par {amt} kharch hue‖ {date} ko‖. Avl limit {limit}",\n            "Card {card} se {amt} ka txn {cp} par‖, {date}‖. Ref {ref}‖. Avl credit limit {limit}",\n            "{bank} Debit Card {card} use hua {cp} par {amt} ke liye‖, {date} {time}‖. Bal {bal}",\n        ],\n    },\n    "atm_withdrawal": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["ATM"], cp=None),\n        "en": [\n            "{amt} withdrawn from A/c {acct}‖ at {inst}‖ on {date}‖. Avl Bal {bal}",\n            "Cash withdrawal{@inst} of {amt} from your {bank} A/c {acct}‖ on {date} {time}‖. Bal: {bal}",\n            "ATM txn{@inst}: {amt} debited from a/c {acct}‖ on {date}‖. Avl bal {bal}",\n        ],\n        "hi": [\n            "खाता {acct} से {amt} की नकद निकासी{@inst}‖ {date} को‖। उपलब्ध शेष {bal}",\n            "{inst} से {amt} निकाले गए, खाता {acct}‖, दिनांक {date}‖। शेष राशि {bal}",\n            "{bank} ATM लेनदेन{@inst}: खाता {acct} से {amt} डेबिट‖, {date} {time}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "A/c {acct} se {amt} cash nikale gaye{@inst}‖ {date} ko‖. Avl bal {bal}",\n            "{inst} se {amt} withdraw hue, A/c {acct}‖, date {date}‖. Bal {bal}",\n            "{bank} ATM txn{@inst}: A/c {acct} se {amt} debit‖, {date} {time}‖. Bal {bal}",\n        ],\n    },\n    "transfer_credit": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=["NEFT", "IMPS", "RTGS"], cp="transfer_party"),\n        "en": [\n            "Your A/c {acct} has been credited with {amt}‖ on {date}‖ by {inst}‖ from {cp}‖. UTR {ref}‖. Avl Bal {bal}",\n            "{inst} credit of {amt} received in A/c {acct} from {cp}‖ on {date}‖. Ref {ref}",\n            "Salary credited: {amt} to A/c {acct}‖ from {cp}‖ via {inst}‖ on {date}‖. Bal {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} में {inst} द्वारा {amt} जमा हुए‖, {cp} से‖, दिनांक {date}‖। UTR {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} से {amt} का {inst} क्रेडिट खाता {acct} में प्राप्त‖ ({date})‖। संदर्भ {ref}",\n            "वेतन जमा: खाता {acct} में {amt}‖, {cp} द्वारा‖ {inst} से‖, {date}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} mein {inst} se {amt} credit hue‖ {cp} se‖, {date} ko‖. UTR {ref}‖. Avl bal {bal}",\n            "{cp} se {amt} ka {inst} credit A/c {acct} mein aaya‖ ({date})‖. Ref {ref}",\n            "Salary aa gayi: A/c {acct} mein {amt}‖ {cp} se‖ {inst} dwara‖, {date}‖. Bal {bal}",\n        ],\n    },\n    "transfer_debit": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["NEFT", "IMPS", "RTGS"], cp="person"),\n        "en": [\n            "{amt} debited from A/c {acct}‖ on {date}‖ for {inst} transfer‖ to {cp}‖. Ref {ref}‖. Avl Bal {bal}",\n            "Your {inst} of {amt} to {cp} from A/c {acct} is successful‖. Ref No {ref}‖. Date {date}",\n            "A/c {acct} debited by {amt}‖ on {date}‖; {inst} to {cp}‖, UTR {ref}",\n        ],\n        "hi": [\n            "खाता {acct} से {amt} डेबिट‖, {date}‖, {inst} ट्रांसफ़र‖ {cp} को‖। संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} को {amt} का आपका {inst} ट्रांसफ़र सफल रहा, खाता {acct}‖। संदर्भ संख्या {ref}‖। दिनांक {date}",\n            "खाता {acct} से {amt} कटे‖ {date} को‖; {cp} को {inst}‖, UTR {ref}",\n        ],\n        "hinglish": [\n            "A/c {acct} se {amt} debit‖, {date}‖, {inst} transfer‖ {cp} ko‖. Ref {ref}‖. Avl bal {bal}",\n            "{cp} ko {amt} ka aapka {inst} transfer successful, A/c {acct}‖. Ref no {ref}‖. Date {date}",\n            "A/c {acct} se {amt} kat gaye‖ {date} ko‖; {cp} ko {inst}‖, UTR {ref}",\n        ],\n    },\n    "refund_reversal": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=[], cp="merchant"),\n        "en": [\n            ("Refund of {amt} from {cp} credited to your A/c {acct}‖ on {date}‖. Avl Bal {bal}", "success"),\n            ("{amt} debited on {date} for txn to {cp} has been reversed to your A/c {acct}‖. Ref {ref}", "reversed"),\n            ("Your refund of {amt} for order at {cp} is processed to Card {card}‖ on {date}", "success"),\n        ],\n        "hi": [\n            ("{cp} से {amt} का रिफ़ंड आपके खाते {acct} में जमा किया गया‖, दिनांक {date}‖। उपलब्ध शेष {bal}", "success"),\n            ("{date} को {cp} के लेनदेन में कटे {amt} आपके खाते {acct} में वापस कर दिए गए हैं‖। संदर्भ {ref}", "reversed"),\n            ("{cp} पर ऑर्डर का {amt} रिफ़ंड कार्ड {card} में भेज दिया गया है‖, {date}", "success"),\n        ],\n        "hinglish": [\n            ("{cp} se {amt} ka refund aapke A/c {acct} mein credit hua‖ {date} ko‖. Avl bal {bal}", "success"),\n            ("{date} ko {cp} txn mein kate {amt} aapke A/c {acct} mein wapas aa gaye‖. Ref {ref}", "reversed"),\n            ("{cp} order ka {amt} refund Card {card} mein bhej diya gaya‖, {date}", "success"),\n        ],\n    },\n    "failed_txn": {\n        "meta": dict(is_txn=True, type="debit", status="failed", inst=["UPI"], cp="merchant"),\n        "en": [\n            "Txn of {amt} to {cp} via {inst} FAILED‖ on {date}‖. Amount, if debited, will be refunded‖. Ref {ref}",\n            "Your payment of {amt} to {cp} could not be processed‖ (UPI Ref {ref}{@inst})‖. A/c {acct} not debited",\n            "Transaction declined: {amt} at {cp} on Card {card}‖ on {date}‖. Reason: insufficient balance",\n        ],\n        "hi": [\n            "{inst} द्वारा {cp} को {amt} का लेनदेन विफल रहा‖, दिनांक {date}‖। अगर राशि कटी है तो वापस आ जाएगी‖। संदर्भ {ref}",\n            "{cp} को {amt} का आपका भुगतान नहीं हो सका‖ (UPI रेफ़ {ref}{@inst})‖। खाता {acct} से राशि नहीं कटी",\n            "लेनदेन अस्वीकृत: कार्ड {card} से {cp} पर {amt}‖, {date}‖। कारण: अपर्याप्त शेष",\n        ],\n        "hinglish": [\n            "{inst} se {cp} ko {amt} ka txn fail ho gaya‖ {date} ko‖. Paisa kata hai to wapas aa jayega‖. Ref {ref}",\n            "{cp} ko {amt} ka aapka payment nahi ho paya‖ (UPI Ref {ref}{@inst})‖. A/c {acct} se paisa nahi kata",\n            "Txn decline hua: Card {card} se {cp} par {amt}‖, {date}‖. Reason: balance kam hai",\n        ],\n    },\n    "autopay_emi": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["NACH", "AUTOPAY"], cp="subscription"),\n        "en": [\n            "{amt} debited from A/c {acct} towards EMI for loan {ref}‖ via {inst}‖ on {date}‖. Avl Bal {bal}",\n            "AutoPay{@inst=AUTOPAY} of {amt} for {cp} executed from A/c {acct}‖ on {date}‖. Ref {ref}",\n            "Your {cp} subscription of {amt} was paid via {inst}‖ from A/c {acct}‖ on {date}",\n        ],\n        "hi": [\n            "लोन {ref} की EMI के लिए खाता {acct} से {amt} डेबिट‖ {inst} द्वारा‖, दिनांक {date}‖। उपलब्ध शेष {bal}",\n            "{cp} के लिए {amt} का ऑटोपे{@inst=AUTOPAY} खाता {acct} से पूरा हुआ‖, {date}‖। संदर्भ {ref}",\n            "आपकी {cp} सदस्यता के {amt} {inst} से चुकाए गए‖, खाता {acct}‖, {date}",\n        ],\n        "hinglish": [\n            "Loan {ref} ki EMI ke liye A/c {acct} se {amt} debit‖ {inst} se‖, {date} ko‖. Avl bal {bal}",\n            "{cp} ke liye {amt} ka AutoPay{@inst=AUTOPAY} A/c {acct} se ho gaya‖, {date}‖. Ref {ref}",\n            "Aapki {cp} subscription ke {amt} {inst} se pay hue‖, A/c {acct}‖, {date}",\n        ],\n    },\n    # ---------------- hard negatives: money mentioned, but nothing moved ----------------\n    "otp": {\n        "meta": dict(is_txn=False, cp="merchant"),\n        "en": [\n            "{otp} is the OTP for your txn of {amt} at {cp} on {bank} Card {card}. Valid for 5 mins. Do not share it with anyone.",\n            "OTP {otp} to authorise payment of {amt} to {cp}. Never share your OTP. -{bank}",\n        ],\n        "hi": [\n            "{otp} आपके {bank} कार्ड {card} से {cp} पर {amt} के लेनदेन का OTP है। 5 मिनट तक मान्य। इसे किसी से साझा न करें।",\n            "{cp} को {amt} के भुगतान के लिए OTP {otp} है। अपना OTP कभी किसी को न बताएँ। -{bank}",\n        ],\n        "hinglish": [\n            "{otp} aapke {bank} Card {card} se {cp} par {amt} ke txn ka OTP hai. 5 min tak valid. Kisi ke saath share na karein.",\n            "{cp} ko {amt} payment ke liye OTP {otp} hai. Apna OTP kabhi kisi ko na batayein. -{bank}",\n        ],\n    },\n    "promo": {\n        "meta": dict(is_txn=False, cp=None),\n        "en": [\n            "Get cashback up to {amt} on your next UPI payment with {bank}! Offer valid till {date}. T&C apply.",\n            "Pre-approved loan of up to {amt} is waiting for you. Apply in the {bank} app today.",\n        ],\n        "hi": [\n            "{bank} के साथ अगले UPI भुगतान पर {amt} तक का कैशबैक पाएँ! ऑफ़र {date} तक मान्य। नियम व शर्तें लागू।",\n            "आपके लिए {amt} तक का प्री-अप्रूव्ड लोन तैयार है। आज ही {bank} ऐप में आवेदन करें।",\n        ],\n        "hinglish": [\n            "{bank} ke saath agle UPI payment par {amt} tak cashback paayein! Offer {date} tak valid. T&C apply.",\n            "Aapke liye {amt} tak ka pre-approved loan ready hai. Aaj hi {bank} app mein apply karein.",\n        ],\n    },\n    "bill_due": {\n        "meta": dict(is_txn=False, cp="biller"),\n        "en": [\n            "Your {bank} Credit Card {card} bill of {amt} is due on {date}. Minimum due {amt2}. Please pay on time to avoid charges.",\n            "Reminder: {cp} bill of {amt} is due on {date}. Ignore if already paid.",\n        ],\n        "hi": [\n            "आपके {bank} क्रेडिट कार्ड {card} का {amt} का बिल {date} को देय है। न्यूनतम देय राशि {amt2}। शुल्क से बचने के लिए समय पर भुगतान करें।",\n            "रिमाइंडर: {cp} का {amt} का बिल {date} को देय है। यदि भुगतान कर दिया है तो इसे अनदेखा करें।",\n        ],\n        "hinglish": [\n            "Aapke {bank} Credit Card {card} ka {amt} ka bill {date} ko due hai. Minimum due {amt2}. Charges se bachne ke liye time par pay karein.",\n            "Reminder: {cp} ka {amt} ka bill {date} ko due hai. Pay kar diya hai to ignore karein.",\n        ],\n    },\n    "scheduled_debit": {\n        "meta": dict(is_txn=False, cp="subscription"),\n        "en": [\n            "{amt} will be debited from A/c {acct} on {date} towards {cp} AutoPay. Please maintain sufficient balance.",\n            "Upcoming EMI of {amt} for loan {ref} will be deducted on {date}. -{bank}",\n        ],\n        "hi": [\n            "{date} को {cp} ऑटोपे के लिए खाता {acct} से {amt} डेबिट किए जाएँगे। कृपया पर्याप्त शेष रखें।",\n            "लोन {ref} की {amt} की आगामी EMI {date} को कटेगी। -{bank}",\n        ],\n        "hinglish": [\n            "{date} ko {cp} AutoPay ke liye A/c {acct} se {amt} debit honge. Kripya sufficient balance rakhein.",\n            "Loan {ref} ki {amt} ki agli EMI {date} ko kategi. -{bank}",\n        ],\n    },\n    "balance_info": {\n        "meta": dict(is_txn=False, cp=None),\n        "en": [\n            "Avl bal in your {bank} A/c {acct} is {bal} as on {date}.",\n            "Dear Customer, the balance in A/c {acct} on {date} is {bal}. -{bank}",\n        ],\n        "hi": [\n            "{date} को आपके {bank} खाते {acct} में उपलब्ध शेष {bal} है।",\n            "प्रिय ग्राहक, खाता {acct} में {date} को शेष राशि {bal} है। -{bank}",\n        ],\n        "hinglish": [\n            "{date} ko aapke {bank} A/c {acct} mein avl bal {bal} hai.",\n            "Dear Customer, A/c {acct} mein {date} ko balance {bal} hai. -{bank}",\n        ],\n    },\n}\nTXN_FAMILIES = [f for f, v in T.items() if v["meta"]["is_txn"]]\nNEG_FAMILIES = [f for f, v in T.items() if not v["meta"]["is_txn"]]\n\n\n# ---------------------------------------------------------------------------\n# Value renderers\n# ---------------------------------------------------------------------------\ndef indian_group(n: int) -> str:\n    s = str(n)\n    if len(s) <= 3:\n        return s\n    head, tail = s[:-3], s[-3:]\n    parts = []\n    while len(head) > 2:\n        parts.insert(0, head[-2:])\n        head = head[:-2]\n    if head:\n        parts.insert(0, head)\n    return ",".join(parts) + "," + tail\n\n\ndef fmt_number(v: float, rng: random.Random) -> str:\n    whole = int(round(v * 100)) // 100\n    paise = int(round(v * 100)) % 100\n    style = rng.choice(["plain", "indian", "indian", "western"])\n    w = str(whole) if style == "plain" else indian_group(whole) if style == "indian" else f"{whole:,}"\n    if paise == 0 and rng.random() < 0.5:\n        return w\n    return f"{w}.{paise:02d}"\n\n\nINR_FORMATS = {\n    "en": ["Rs.{n}", "Rs {n}", "INR {n}", "₹{n}", "Rs.{n}/-", "INR{n}"],\n    "hi": ["₹{n}", "रु. {n}", "रुपये {n}", "Rs.{n}", "₹ {n}"],\n    "hinglish": ["Rs {n}", "Rs.{n}", "₹{n}", "{n} rupaye", "INR {n}"],\n}\n\n\ndef rand_amount(rng: random.Random, lo=1.0, hi=500000.0) -> float:\n    import math\n    v = math.exp(rng.uniform(math.log(lo), math.log(hi)))\n    return round(v, 2) if rng.random() < 0.6 else float(int(v))\n\n\ndef fmt_inr(v: float, lang: str, rng: random.Random) -> str:\n    return rng.choice(INR_FORMATS[lang]).format(n=fmt_number(v, rng))\n\n\ndef fmt_foreign(v: float, code: str, sym: str, rng: random.Random) -> str:\n    n = f"{v:.2f}"\n    return rng.choice([f"{code} {n}", f"{sym}{n}" if sym.strip() != code else f"{code} {n}", f"{code}{n}"])\n\n\ndef fmt_date(d: date, rng: random.Random) -> str:\n    style = rng.randrange(8)\n    m = MONTHS[d.month - 1]\n    return [\n        f"{d.day:02d}-{d.month:02d}-{d.year % 100:02d}",\n        f"{d.day:02d}/{d.month:02d}/{d.year}",\n        f"{d.day:02d}{m}{d.year % 100:02d}",\n        f"{d.day:02d}-{m}-{d.year}",\n        f"{d.year}-{d.month:02d}-{d.day:02d}",\n        f"{d.day:02d}.{d.month:02d}.{d.year}",\n        f"{d.day} {m} {d.year}",\n        f"{d.day:02d}-{m.upper()}-{d.year % 100:02d}",\n    ][style]\n\n\ndef fmt_acct(l4: str, lang: str, rng: random.Random) -> str:\n    opts = [f"XX{l4}", f"XXXXXX{l4}", f"*{l4}", f"x{l4}", f"XXXX{l4}", f"**{l4}"]\n    if lang == "en":\n        opts += [f"no. XX{l4}", f"ending {l4}"]\n    return rng.choice(opts)\n\n\ndef fmt_card(l4: str, rng: random.Random) -> str:\n    return rng.choice([f"XX{l4}", f"ending {l4}", f"**** {l4}", f"xx{l4}", f"XXXX-{l4}"])\n\n\ndef rand_ref(inst: str | None, rng: random.Random) -> str:\n    if inst in ("NEFT", "RTGS") and rng.random() < 0.6:\n        return rng.choice("NHSKBP") + "".join(rng.choice("0123456789") for _ in range(15))\n    if rng.random() < 0.15:\n        return "".join(rng.choice("ABCDEFGHJKLMNPQRSTUVWXYZ0123456789") for _ in range(10))\n    return "".join(rng.choice("0123456789") for _ in range(12))\n\n\ndef rand_counterparty(pool: str | None, rng: random.Random) -> str | None:\n    if pool is None:\n        return None\n    if pool == "upi_party":\n        r = rng.random()\n        if r < 0.4:\n            return rng.choice(MERCHANTS)\n        if r < 0.7:\n            return rng.choice(PERSONS)\n        name = rng.choice(PERSONS).lower().split()\n        return f"{name[0]}.{name[1][:1]}{rng.randint(1, 99)}{rng.choice(VPA_HANDLES)}"\n    if pool == "transfer_party":\n        return rng.choice(EMPLOYERS) if rng.random() < 0.5 else rng.choice(PERSONS)\n    return rng.choice({"merchant": MERCHANTS, "person": PERSONS, "subscription": SUBSCRIPTIONS,\n                       "biller": BILLERS}[pool])\n\n\n# ---------------------------------------------------------------------------\n# Perturbations (edge cases). Each returns (text, tag) and never changes gold values.\n# ---------------------------------------------------------------------------\nDEV_DIGITS = str.maketrans("0123456789", "०१२३४५६७८९")\n\n\ndef perturb(text: str, lang: str, rng: random.Random, p: float = 0.35) -> tuple[str, list[str]]:\n    tags = []\n    if rng.random() > p:\n        return text, tags\n    ops = ["spacing", "header", "promo_tail", "footer"]\n    if lang in ("en", "hinglish"):\n        ops += ["upper", "lower"]\n    if lang == "hi":\n        ops += ["dev_digits"]\n    for op in rng.sample(ops, k=rng.choice([1, 1, 2])):\n        if op == "upper":\n            text = text.upper()\n        elif op == "lower":\n            text = text.lower()\n        elif op == "spacing":\n            # remove a space between a number and a letter, or double a space\n            if rng.random() < 0.5:\n                text = re.sub(r"(?<=\\d) (?=[^\\W\\d])", "", text, count=1)\n            else:\n                text = re.sub(r" ", "  ", text, count=1)\n        elif op == "header":\n            text = rng.choice(["[Kestrel Bk] ", "BNYN: ", "Alert: ", "Txn Alert- "]) + text\n        elif op == "footer":\n            text = text + rng.choice([" -Kestrel Bank", " Not you? Report via your bank app.",\n                                      " Download our app for details.", " - Neem Bank"])\n        elif op == "promo_tail":\n            text = text + rng.choice([" Get 5% cashback up to Rs.100 on your next txn!",\n                                      " Earn 2X reward points on weekend spends.",\n                                      " Win vouchers worth Rs 500. T&C apply."])\n        elif op == "dev_digits":\n            text = text.translate(DEV_DIGITS)\n        tags.append(op)\n    return text, tags\n\n\n# ---------------------------------------------------------------------------\n# Rendering one example\n# ---------------------------------------------------------------------------\n@dataclass\nclass Example:\n    id: str\n    family: str\n    lang: str\n    template_id: str\n    sms: str\n    gold: dict\n    tags: list = field(default_factory=list)\n    split: str = ""\n\n    def to_json(self) -> dict:\n        return {"id": self.id, "family": self.family, "lang": self.lang,\n                "template_id": self.template_id, "sms": self.sms,\n                "gold": self.gold, "tags": self.tags, "split": self.split}\n\n\n_SLOT = re.compile(r"\\{(@?)(\\w+)(?:=(\\w+))?\\}")\n\n\ndef render(family: str, lang: str, t_idx: int, rng: random.Random, ex_id: str,\n           noise_p: float = 0.35, allow_drop: bool = True) -> Example:\n    spec = T[family]\n    meta = spec["meta"]\n    tmpl = spec[lang][t_idx]\n    status = meta.get("status")\n    if isinstance(tmpl, tuple):\n        tmpl, status = tmpl\n    segments = tmpl.split("‖")\n    keep = [True] * len(segments)\n    tags = []\n    if allow_drop and meta["is_txn"] and len(segments) > 1:\n        # optional-segment dropping (30%) and truncation (10%)\n        for k in range(1, len(segments)):\n            if rng.random() < 0.15:\n                keep[k] = False\n        if any(not x for x in keep[1:]):\n            tags.append("missing_fields")\n        if rng.random() < 0.10:\n            cut = rng.randint(1, len(segments) - 1)\n            for k in range(cut, len(segments)):\n                keep[k] = False\n            tags.append("truncated")\n\n    inst = rng.choice(meta["inst"]) if meta.get("inst") else None\n    foreign = meta.get("foreign", 0) and rng.random() < meta["foreign"]\n    amount = rand_amount(rng, 1, 3000) if foreign else rand_amount(rng)\n    code, sym = rng.choice(FOREIGN) if foreign else ("INR", "₹")\n    d = date(2025, 1, 1) + timedelta(days=rng.randrange(0, 730))\n    acct = f"{rng.randint(0, 9999):04d}"\n    card = f"{rng.randint(0, 9999):04d}"\n    values = {\n        "amt": fmt_foreign(amount, code, sym, rng) if foreign else fmt_inr(amount, lang, rng),\n        "amt2": fmt_inr(round(amount * 0.05, 2) if amount > 100 else amount, lang, rng),\n        "acct": fmt_acct(acct, lang, rng),\n        "card": fmt_card(card, rng),\n        "date": fmt_date(d, rng),\n        "time": f"{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}",\n        "cp": rand_counterparty(meta.get("cp"), rng) or "",\n        "ref": rand_ref(inst, rng),\n        "bal": fmt_inr(rand_amount(rng, 10, 900000), lang, rng),\n        "limit": fmt_inr(rand_amount(rng, 1000, 500000), lang, rng),\n        "inst": {"AUTOPAY": "AutoPay"}.get(inst, inst) if inst else "",\n        "bank": rng.choice(FICTIONAL_BANKS),\n        "otp": f"{rng.randint(0, 999999):06d}",\n    }\n    gold_values = {\n        "amt": ("amount", amount), "acct": ("account_last4", acct), "card": ("account_last4", card),\n        "date": ("date", d.isoformat()), "cp": ("counterparty", values["cp"] or None),\n        "ref": ("reference", values["ref"]), "bal": ("balance", None),\n        "inst": ("instrument", inst),\n    }\n    # balance gold value is the numeric value we rendered; re-derive from text below\n    bal_value = None\n\n    revealed = {}\n    parts = []\n    for seg, k in zip(segments, keep):\n        if not k:\n            continue\n\n        def sub(m):\n            hidden, name, fixed = m.group(1), m.group(2), m.group(3)\n            if name in gold_values:\n                fname, val = gold_values[name]\n                if fixed:\n                    val = fixed\n                revealed[fname] = val\n                if name == "card":\n                    revealed["instrument"] = "CARD"\n                if name == "amt":\n                    revealed["currency"] = code\n            if name == "bal":\n                revealed["balance"] = "__BAL__"\n            return "" if hidden else values[name]\n        parts.append(_SLOT.sub(sub, seg))\n    text = "".join(parts).strip()\n    # numeric balance value from rendered string\n    if "balance" in revealed:\n        bal_value = parse_rendered_number(values["bal"])\n        revealed["balance"] = bal_value\n    if text and text[-1] not in ".।!":\n        text += "." if lang != "hi" else "।"\n\n    gold = {f: None for f in FIELDS}\n    gold["is_transaction"] = bool(meta["is_txn"])\n    if meta["is_txn"]:\n        gold["type"] = meta["type"]\n        gold["status"] = status\n        for k, v in revealed.items():\n            gold[k] = v\n        if foreign:\n            tags.append("foreign_currency")\n        if "{limit}" in "".join(s for s, k in zip(segments, keep) if k):\n            tags.append("limit_distractor")\n    else:\n        tags.append("hard_negative")\n    text, ptags = perturb(text, lang, rng, p=noise_p)\n    tags += ptags\n    return Example(id=ex_id, family=family, lang=lang, template_id=f"{family}/{lang}/{t_idx}",\n                   sms=text, gold=gold, tags=tags)\n\n\n_NUM = re.compile(r"[\\d,]+(?:\\.\\d+)?")\n\n\ndef parse_rendered_number(s: str) -> float:\n    m = _NUM.search(s)\n    return float(m.group(0).replace(",", "")) if m else None\n\n\n# ---------------------------------------------------------------------------\n# Dataset builder with template-level held-out split\n# ---------------------------------------------------------------------------\ndef build_dataset(n_train=2400, n_dev=200, n_test_id=400, n_test_ood=400, seed=7,\n                  neg_share=0.3, noise_p=0.35, allow_drop=True,\n                  exclude_sms: set | None = None) -> dict[str, list[Example]]:\n    """noise_p / allow_drop / neg_share = 0 give the \'clean data\' ablation."""\n    rng = random.Random(seed)\n    seen_sms = set(exclude_sms or ())\n    out = {"train": [], "dev": [], "test_id": [], "test_ood": []}\n\n    def pick(held_out: bool):\n        fam = rng.choice(NEG_FAMILIES) if rng.random() < neg_share else rng.choice(TXN_FAMILIES)\n        lang = rng.choice(LANGS)\n        n_t = len(T[fam][lang])\n        t_idx = n_t - 1 if held_out else rng.randrange(0, n_t - 1)\n        return fam, lang, t_idx\n\n    for split, n in [("test_ood", n_test_ood), ("test_id", n_test_id), ("dev", n_dev), ("train", n_train)]:\n        i = 0\n        tries = 0\n        while len(out[split]) < n:\n            tries += 1\n            fam, lang, t_idx = pick(held_out=(split == "test_ood"))\n            ex = render(fam, lang, t_idx, rng, f"{split}-{i:05d}", noise_p=noise_p,\n                        allow_drop=allow_drop)\n            if ex.sms in seen_sms:      # exact-duplicate guard across ALL splits (no leakage)\n                continue\n            seen_sms.add(ex.sms)\n            ex.split = split\n            out[split].append(ex)\n            i += 1\n    return out\n\n\ndef gold_to_str(gold: dict) -> str:\n    return json.dumps({k: gold[k] for k in FIELDS}, ensure_ascii=False)\n\n\nSYSTEM_PROMPT = (\n    "You extract structured data from Indian bank/UPI SMS (English, Hindi or Hinglish). "\n    "Reply with ONLY one JSON object with exactly these keys: "\n    "is_transaction (true only if money actually moved or a transaction was attempted/failed/reversed; "\n    "false for OTPs, offers, reminders, future debits, balance info), "\n    "type (\\"debit\\"|\\"credit\\"|null), amount (number), currency (ISO code, e.g. \\"INR\\"), "\n    "account_last4 (last 4 digits as string), "\n    "instrument (\\"UPI\\"|\\"CARD\\"|\\"NEFT\\"|\\"IMPS\\"|\\"RTGS\\"|\\"ATM\\"|\\"NACH\\"|\\"AUTOPAY\\"|null), "\n    "counterparty (merchant/person/VPA as written), date (YYYY-MM-DD; SMS dates are day-first), "\n    "reference (string), balance (number; available balance only, never credit limit), "\n    "status (\\"success\\"|\\"failed\\"|\\"reversed\\"|null). "\n    "Use null for anything not stated in the SMS. Never guess. Use ASCII digits. "\n    "If is_transaction is false, set every other key to null."\n)\n\n\ndef build_messages(sms: str, few_shot: list[tuple[str, str]] | None = None) -> list[dict]:\n    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]\n    for s, g in (few_shot or []):\n        msgs.append({"role": "user", "content": f"SMS: {s}"})\n        msgs.append({"role": "assistant", "content": g})\n    msgs.append({"role": "user", "content": f"SMS: {sms}"})\n    return msgs\n\n\nif __name__ == "__main__":\n    ds = build_dataset()\n    for k, v in ds.items():\n        print(k, len(v))\n    for ex in ds["train"][:6]:\n        print(ex.lang, ex.family, ex.tags, "\\n ", ex.sms, "\\n ", gold_to_str(ex.gold))\n'
open(f"{WORK}/src/smsgen.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/smsgen.py', len(CONTENT), 'chars')

In [ ]:
# ---- src/scoring.py ----
CONTENT = '"""Parsing, schema validation and metrics for SMS -> JSON extraction."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport random\nimport re\nimport unicodedata\nfrom collections import defaultdict\n\nFIELDS = ["is_transaction", "type", "amount", "currency", "account_last4", "instrument",\n          "counterparty", "date", "reference", "balance", "status"]\nVALUE_FIELDS = FIELDS[1:]\nENUMS = {\n    "type": {"debit", "credit"},\n    "instrument": {"UPI", "CARD", "NEFT", "IMPS", "RTGS", "ATM", "NACH", "AUTOPAY"},\n    "status": {"success", "failed", "reversed"},\n}\n_DEV = str.maketrans("०१२३४५६७८९", "0123456789")\n\n\n# ---------------------------------------------------------------------------\n# Parsing\n# ---------------------------------------------------------------------------\ndef extract_json(text: str) -> dict | None:\n    """Return the first JSON object in `text` (tolerates code fences / chatter)."""\n    if not text:\n        return None\n    t = text.strip()\n    t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    start = t.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(t)):\n            c = t[i]\n            if in_str:\n                if esc:\n                    esc = False\n                elif c == "\\\\":\n                    esc = True\n                elif c == \'"\':\n                    in_str = False\n            elif c == \'"\':\n                in_str = True\n            elif c == "{":\n                depth += 1\n            elif c == "}":\n                depth -= 1\n                if depth == 0:\n                    try:\n                        obj = json.loads(t[start:i + 1])\n                        return obj if isinstance(obj, dict) else None\n                    except json.JSONDecodeError:\n                        break\n        start = t.find("{", start + 1)\n    return None\n\n\ndef schema_errors(obj: dict | None) -> list[str]:\n    if obj is None:\n        return ["not_json"]\n    errs = []\n    keys = set(obj)\n    if keys != set(FIELDS):\n        if set(FIELDS) - keys:\n            errs.append("missing_keys")\n        if keys - set(FIELDS):\n            errs.append("extra_keys")\n    if not isinstance(obj.get("is_transaction"), bool):\n        errs.append("bad_is_transaction")\n    for k, allowed in ENUMS.items():\n        v = obj.get(k)\n        if v is not None and (not isinstance(v, str) or v not in allowed):\n            errs.append(f"bad_{k}")\n    for k in ("amount", "balance"):\n        v = obj.get(k)\n        if v is not None and not (isinstance(v, (int, float)) and not isinstance(v, bool)):\n            errs.append(f"bad_{k}")\n    d = obj.get("date")\n    if d is not None and not (isinstance(d, str) and re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", d)):\n        errs.append("bad_date")\n    a = obj.get("account_last4")\n    if a is not None and not (isinstance(a, str) and re.fullmatch(r"\\d{4}", a)):\n        errs.append("bad_account_last4")\n    return errs\n\n\n# ---------------------------------------------------------------------------\n# Normalisation & field comparison (lenient on formatting, strict on meaning)\n# ---------------------------------------------------------------------------\ndef _num(v):\n    if v is None:\n        return None\n    if isinstance(v, bool):\n        return None\n    if isinstance(v, (int, float)):\n        return float(v)\n    s = str(v).translate(_DEV)\n    m = re.search(r"\\d[\\d,]*(?:\\.\\d+)?", s)\n    return float(m.group(0).replace(",", "")) if m else None\n\n\ndef _norm_str(v):\n    if v is None:\n        return None\n    s = unicodedata.normalize("NFC", str(v)).translate(_DEV).casefold()\n    s = re.sub(r"\\s+", " ", s).strip()\n    return s or None\n\n\ndef _norm_alnum(v):\n    s = _norm_str(v)\n    if not s:\n        return None\n    s = "".join(ch for ch in s if ch in "@." or unicodedata.category(ch)[0] in "LMN")   # keeps Devanagari letters\n    return s or None\n\n\ndef norm_field(k, v):\n    if v is None or (isinstance(v, str) and v.strip().lower() in ("", "null", "none", "n/a")):\n        return None\n    if k in ("amount", "balance"):\n        return _num(v)\n    if k == "account_last4":\n        d = re.sub(r"\\D", "", str(v).translate(_DEV))\n        return d[-4:] if d else None\n    if k in ("instrument", "currency"):\n        return _norm_str(v).upper() if _norm_str(v) else None\n    if k in ("type", "status"):\n        return _norm_str(v)\n    if k in ("reference", "counterparty"):\n        return _norm_alnum(v)\n    if k == "date":\n        return _norm_str(v)\n    if k == "is_transaction":\n        if isinstance(v, bool):\n            return v\n        return {"true": True, "false": False}.get(str(v).lower())\n    return _norm_str(v)\n\n\ndef field_equal(k, pred, gold) -> bool:\n    p, g = norm_field(k, pred), norm_field(k, gold)\n    if g is None or p is None:\n        return g is None and p is None\n    if k in ("amount", "balance"):\n        return abs(p - g) < 0.005\n    return p == g\n\n\n# ---------------------------------------------------------------------------\n# Per-example and aggregate scoring\n# ---------------------------------------------------------------------------\ndef score_example(raw_output: str, gold: dict) -> dict:\n    obj = extract_json(raw_output)\n    errs = schema_errors(obj)\n    pred = obj or {}\n    res = {"json_parsed": obj is not None, "schema_valid": obj is not None and not errs,\n           "schema_errors": errs}\n    correct = {}\n    for k in FIELDS:\n        correct[k] = field_equal(k, pred.get(k), gold.get(k))\n    res["field_correct"] = correct\n    res["exact_match"] = obj is not None and all(correct.values())\n    # hallucination: predicted a value where gold is null; omission: the reverse\n    hall = omit = gold_null = gold_val = 0\n    for k in VALUE_FIELDS:\n        g = norm_field(k, gold.get(k))\n        p = norm_field(k, pred.get(k))\n        if g is None:\n            gold_null += 1\n            hall += p is not None\n        else:\n            gold_val += 1\n            omit += p is None\n    res.update(hallucinated=hall, gold_null=gold_null, omitted=omit, gold_valued=gold_val)\n    res["pred_is_txn"] = norm_field("is_transaction", pred.get("is_transaction"))\n    res["gold_is_txn"] = gold["is_transaction"]\n    return res\n\n\ndef aggregate(scored: list[dict]) -> dict:\n    n = len(scored)\n    if n == 0:\n        return {}\n    out = {\n        "n": n,\n        "json_valid": sum(s["json_parsed"] for s in scored) / n,\n        "schema_valid": sum(s["schema_valid"] for s in scored) / n,\n        "exact_match": sum(s["exact_match"] for s in scored) / n,\n    }\n    # micro field accuracy over value fields on transaction examples\n    txn = [s for s in scored if s["gold_is_txn"]]\n    if txn:\n        out["field_acc_txn"] = sum(s["field_correct"][k] for s in txn for k in VALUE_FIELDS) / (\n            len(txn) * len(VALUE_FIELDS))\n        for k in VALUE_FIELDS:\n            out[f"acc_{k}"] = sum(s["field_correct"][k] for s in txn) / len(txn)\n    gn = sum(s["gold_null"] for s in scored)\n    gv = sum(s["gold_valued"] for s in scored)\n    out["hallucination_rate"] = sum(s["hallucinated"] for s in scored) / gn if gn else float("nan")\n    out["omission_rate"] = sum(s["omitted"] for s in scored) / gv if gv else float("nan")\n    # transaction detection (positive class = NOT a transaction, i.e. catching hard negatives)\n    tp = sum(1 for s in scored if s["gold_is_txn"] is False and s["pred_is_txn"] is False)\n    fp = sum(1 for s in scored if s["gold_is_txn"] is True and s["pred_is_txn"] is False)\n    fn = sum(1 for s in scored if s["gold_is_txn"] is False and s["pred_is_txn"] is not False)\n    prec = tp / (tp + fp) if tp + fp else float("nan")\n    rec = tp / (tp + fn) if tp + fn else float("nan")\n    out["nontxn_precision"], out["nontxn_recall"] = prec, rec\n    out["nontxn_f1"] = 2 * prec * rec / (prec + rec) if prec and rec and not math.isnan(prec + rec) else 0.0\n    out["is_txn_acc"] = sum(1 for s in scored if s["pred_is_txn"] == s["gold_is_txn"]) / n\n    return out\n\n\ndef breakdown(rows: list[dict], scored: list[dict], key: str) -> dict:\n    groups = defaultdict(list)\n    for r, s in zip(rows, scored):\n        vals = r.get(key)\n        for v in (vals if isinstance(vals, list) else [vals]):\n            groups[v].append(s)\n    return {g: aggregate(v) for g, v in sorted(groups.items(), key=lambda x: str(x[0]))}\n\n\n# ---------------------------------------------------------------------------\n# Statistics\n# ---------------------------------------------------------------------------\ndef mcnemar_exact(a: list[bool], b: list[bool]) -> dict:\n    """Paired exact McNemar test (a = system A correct, b = system B correct)."""\n    n01 = sum(1 for x, y in zip(a, b) if not x and y)\n    n10 = sum(1 for x, y in zip(a, b) if x and not y)\n    n = n01 + n10\n    if n == 0:\n        return {"b_only": n01, "a_only": n10, "p_value": 1.0}\n    k = min(n01, n10)\n    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n\n    return {"b_only": n01, "a_only": n10, "p_value": min(1.0, 2 * p)}\n\n\ndef bootstrap_ci(values: list[float], n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:\n    rng = random.Random(seed)\n    n = len(values)\n    if n == 0:\n        return (float("nan"), float("nan"))\n    means = sorted(sum(values[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))\n    return means[int(0.025 * n_boot)], means[int(0.975 * n_boot) - 1]\n'
open(f"{WORK}/src/scoring.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/scoring.py', len(CONTENT), 'chars')

In [ ]:
# ---- src/stage2_data.py ----
CONTENT = '"""Stage 2 data tools: LLM rewrites of training SMS, rephrased test sets, an unrelated-text set,\nand the checks that keep labels correct.\n\nA rewrite changes the wording of an SMS but must keep every fact. Because the gold label is\nreused unchanged, a rewrite is only accepted if the facts can be verified in its text:\n\n* fact gate      - every non-null gold value (amount, balance, account, reference, counterparty,\n                   date, currency, instrument keyword) is still present, and no new number appears\n* meaning gate   - a rule-based reading (regex baseline) of the rewrite agrees with the original\n                   on is_transaction / type / status, so a debit cannot silently become a credit\n* language gate  - Hindi stays in Devanagari, English/Hinglish stay in Latin script\n* hygiene        - length bounds, no links, not identical to the source or to any other split\n"""\nfrom __future__ import annotations\n\nimport json\nimport random\nimport re\nimport unicodedata\n\nimport regex_baseline as RB\n\nDEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")\nNUM_RE = re.compile(r"\\d[\\d,]*(?:\\.\\d+)?")\nURL_RE = re.compile(r"https?://|www\\.|\\.com\\b|\\.in/|bit\\.ly", re.I)\nDEVANAGARI_RE = re.compile(r"[ऀ-ॿ]")\nLATIN_RE = re.compile(r"[A-Za-z]")\n\nINSTRUMENT_KEYWORDS = {\n    "UPI": ["upi", "यूपीआई", "vpa", "@"],\n    "CARD": ["card", "कार्ड"],\n    "ATM": ["atm", "एटीएम"],\n    "NEFT": ["neft", "एनईएफटी"],\n    "IMPS": ["imps", "आईएमपीएस"],\n    "RTGS": ["rtgs", "आरटीजीएस"],\n    "NACH": ["nach", "ecs", "mandate", "मैंडेट"],\n    "AUTOPAY": ["autopay", "auto-pay", "auto pay", "ऑटोपे", "standing instruction", "si "],\n}\nFOREIGN = {"USD": ["usd", "$"], "EUR": ["eur", "€"], "GBP": ["gbp", "£"], "AED": ["aed", "dhs"],\n           "SGD": ["sgd", "s$"]}\n\n\ndef norm(t: str) -> str:\n    t = unicodedata.normalize("NFKC", t or "").translate(DEV_DIGITS)\n    return re.sub(r"\\s+", " ", t).strip()\n\n\ndef numbers(t: str) -> list[float]:\n    out = []\n    for m in NUM_RE.finditer(norm(t)):\n        s = m.group(0).replace(",", "").rstrip(".")\n        try:\n            out.append(float(s))\n        except ValueError:\n            pass\n    return out\n\n\ndef has_number(t: str, v: float) -> bool:\n    return any(abs(n - float(v)) < 0.005 for n in numbers(t))\n\n\ndef _date_tokens(sms: str) -> list[str]:\n    t = norm(sms)\n    pats = [r"\\d{4}-\\d{2}-\\d{2}", r"\\d{1,2}[-/.]\\d{1,2}[-/.]\\d{2,4}",\n            r"\\d{1,2}[-/. ]?[A-Za-z]{3,9}[-/., ]{0,2}\\d{2,4}"]\n    toks = []\n    for p in pats:\n        toks += [m.group(0) for m in re.finditer(p, t)]\n    return toks\n\n\ndef fact_gate(orig_sms: str, new_sms: str, gold: dict) -> tuple[bool, str]:\n    """True if every gold fact is still verifiable in new_sms and no new number was introduced."""\n    o, n = norm(orig_sms), norm(new_sms)\n    nl = n.casefold()\n    for k in ("amount", "balance"):\n        if gold.get(k) is not None and not has_number(n, gold[k]):\n            return False, f"lost_{k}"\n    for k in ("account_last4", "reference"):\n        v = gold.get(k)\n        if v is not None and norm(str(v)).casefold() not in nl:\n            return False, f"lost_{k}"\n    cp = gold.get("counterparty")\n    if cp is not None and norm(cp).casefold() not in nl:\n        return False, "lost_counterparty"\n    if gold.get("date") is not None:\n        same_parse = RB._date(n) == gold["date"]\n        verbatim = any(tok.casefold() in nl for tok in _date_tokens(o))\n        if not (same_parse or verbatim):\n            return False, "lost_date"\n    cur = gold.get("currency")\n    if cur and cur != "INR":\n        if not any(s in nl for s in FOREIGN.get(cur, [cur.lower()])):\n            return False, "lost_currency"\n    else:\n        ol = o.casefold()\n        for code, syms in FOREIGN.items():\n            if any(s in nl and s not in ol for s in syms if s not in ("$",)):\n                return False, f"new_currency_{code}"\n    inst = gold.get("instrument")\n    if inst:\n        kws = [w for w in INSTRUMENT_KEYWORDS.get(inst, []) if w in o.casefold()]\n        if kws and not any(w in nl for w in kws):\n            return False, "lost_instrument"\n    orig_nums = numbers(o)\n    for v in numbers(n):\n        if v >= 10 and not any(abs(v - x) < 0.005 for x in orig_nums):\n            return False, "new_number"\n    return True, "ok"\n\n\ndef meaning_gate(orig_sms: str, new_sms: str) -> tuple[bool, str]:\n    """Rule-based reading must agree on is_transaction / type / status."""\n    a, b = RB.parse(orig_sms), RB.parse(new_sms)\n    for k in ("is_transaction", "type", "status"):\n        if a.get(k) != b.get(k):\n            return False, f"meaning_changed_{k}"\n    return True, "ok"\n\n\ndef language_gate(lang: str, new_sms: str) -> tuple[bool, str]:\n    dev, lat = len(DEVANAGARI_RE.findall(new_sms)), len(LATIN_RE.findall(new_sms))\n    letters = max(dev + lat, 1)\n    if lang == "hi" and dev / letters < 0.3:\n        return False, "lang_not_devanagari"\n    if lang == "en" and dev / letters > 0.05:\n        return False, "lang_has_devanagari"\n    if lang == "hinglish" and dev / letters > 0.25:\n        return False, "lang_mostly_devanagari"\n    return True, "ok"\n\n\ndef hygiene_gate(orig_sms: str, new_sms: str, existing: set[str]) -> tuple[bool, str]:\n    n = norm(new_sms)\n    if not (15 <= len(n) <= 420):\n        return False, "bad_length"\n    if URL_RE.search(n):\n        return False, "has_link"\n    if n.casefold() == norm(orig_sms).casefold():\n        return False, "unchanged"\n    if n.casefold() in existing:\n        return False, "duplicate_or_leak"\n    return True, "ok"\n\n\ndef check_rewrite(row: dict, new_sms: str, existing: set[str]) -> tuple[bool, str]:\n    """Run all gates in a fixed order; return (accepted, first failing reason)."""\n    for gate in (lambda: hygiene_gate(row["sms"], new_sms, existing),\n                 lambda: language_gate(row["lang"], new_sms),\n                 lambda: fact_gate(row["sms"], new_sms, row["gold"]),\n                 lambda: meaning_gate(row["sms"], new_sms)):\n        ok, why = gate()\n        if not ok:\n            return False, why\n    return True, "ok"\n\n\ndef clean_llm_text(t: str) -> str:\n    """Strip quotes, labels and extra lines an LLM may wrap around the SMS."""\n    t = (t or "").strip()\n    t = re.sub(r"^```[a-z]*\\s*|\\s*```$", "", t).strip()\n    t = re.sub(r"^(rewritten|rephrased|paraphrased)?\\s*(sms|message)\\s*:\\s*", "", t, flags=re.I).strip()\n    lines = [l for l in t.splitlines() if l.strip()]\n    t = lines[0].strip() if lines else ""\n    if len(t) >= 2 and t[0] == t[-1] and t[0] in "\\"\'“”":\n        t = t[1:-1].strip()\n    return t.strip("“”\\"").strip()\n\n\nLANG_NAME = {"en": "English", "hi": "Hindi written in Devanagari script",\n             "hinglish": "Hinglish (Hindi written in Latin/Roman script, mixed with English)"}\n\nSTYLES = [\n    "terse, heavily abbreviated (A/c, Txn, Ref, Bal), like a busy bank\'s alert",\n    "formal and polite, starting with \'Dear Customer\' or its equivalent",\n    "put the amount first, then the account, then the other details",\n    "UPI-app notification style, short and friendly",\n    "the bank name at the end after a dash, details in a different order",\n    "slightly longer, adding a generic closing line such as \'not you? contact your bank\' (no phone numbers, no links)",\n]\n\nTRAIN_REWRITE_SYSTEM = (\n    "You rewrite Indian bank and UPI SMS so that they look like a different bank sent them. "\n    "Rules: (1) Copy every number, amount, account digit, card digit, reference/UTR number, date and "\n    "merchant/person name EXACTLY as written, character for character. (2) Keep the meaning exactly: "\n    "a debit stays a debit, a credit stays a credit, a failed or reversed transaction stays failed or "\n    "reversed, an OTP/offer/reminder/future debit/balance alert stays that and must not become a "\n    "transaction. (3) Keep the same language and script. (4) Do not add any new numbers, links, phone "\n    "numbers or details. (5) Reply with the rewritten SMS only, on one line, without quotes."\n)\n\nTEST_REPHRASE_SYSTEM = (\n    "Paraphrase the SMS below. Change the wording and sentence order the way another bank might phrase "\n    "it, but keep every fact. Every number, amount, account or card digit, reference number, date and "\n    "name must appear exactly as in the original. Do not change what happened (debit/credit, success/"\n    "failure, OTP or offer stays the same kind of message). Keep the language and script. Add nothing "\n    "new. Output only the paraphrased SMS on one line."\n)\n\n\ndef train_rewrite_messages(row: dict, rng: random.Random) -> list[dict]:\n    style = rng.choice(STYLES)\n    user = (f"Language: {LANG_NAME.get(row[\'lang\'], row[\'lang\'])}\\nStyle: {style}\\n\\n"\n            f"Original SMS:\\n{row[\'sms\']}")\n    return [{"role": "system", "content": TRAIN_REWRITE_SYSTEM}, {"role": "user", "content": user}]\n\n\ndef test_rephrase_messages(row: dict) -> list[dict]:\n    user = f"Language: {LANG_NAME.get(row[\'lang\'], row[\'lang\'])}\\n\\nSMS:\\n{row[\'sms\']}"\n    return [{"role": "system", "content": TEST_REPHRASE_SYSTEM}, {"role": "user", "content": user}]\n\n\nNULL_GOLD = {"is_transaction": False, "type": None, "amount": None, "currency": None,\n             "account_last4": None, "instrument": None, "counterparty": None, "date": None,\n             "reference": None, "balance": None, "status": None}\n\n\ndef sentences(text: str, lo: int = 30, hi: int = 220) -> list[str]:\n    parts = re.split(r"(?<=[.!?।])\\s+", re.sub(r"\\s+", " ", text or "").strip())\n    return [p for p in parts if lo <= len(p) <= hi]\n'
open(f"{WORK}/src/stage2_data.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/stage2_data.py', len(CONTENT), 'chars')

In [ ]:
# ---- src/regex_baseline.py ----
CONTENT = '"""A reasonable hand-written rule-based parser: the \'no ML\' baseline.\n\nIt is deliberately decent (not a straw man): it handles the common English formats, the\nusual Hindi/Hinglish keywords, Indian digit grouping and day-first dates.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\n\nMONTHS = {m: i + 1 for i, m in enumerate(\n    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}\n_DEV = str.maketrans("०१२३४५६७८९", "0123456789")\n\nNEG_HINTS = ["otp", "will be debited", "due on", "cashback", "pre-approved", "offer",\n             "debit honge", "डेबिट किए जाएँगे", "कटेगी", "kategi", "देय है", "due hai",\n             "upcoming", "आगामी", "agli emi", "balance in a/c", "avl bal in your", "शेष राशि", "उपलब्ध शेष"]\nDEBIT_HINTS = ["debited", "debit", "spent", "withdrawn", "sent", "paid", "डेबिट", "खर्च", "निकाले",\n               "निकासी", "भेजे गए", "कटे", "kharch", "nikale", "bheje gaye", "kat gaye", "withdraw",\n               "used for", "चुकाए", "pay hue", "declined", "failed", "विफल", "fail", "nahi ho"]\nCREDIT_HINTS = ["credited", "received", "refund", "reversed", "जमा", "प्राप्त", "रिफ़ंड", "वापस",\n                "credit hue", "mile", "aaya", "wapas", "bheje{@}", "ने आपको", "ne aapko", "salary"]\nAMT_RE = re.compile(\n    r"(?:(?:rs\\.?|inr|₹|रु\\.?|रुपये|usd|eur|gbp|aed|\\$|€|£)\\s*([\\d,]+(?:\\.\\d+)?))|(?:([\\d,]+(?:\\.\\d+)?)\\s*rupaye)",\n    re.I)\n\n\ndef _num(s):\n    return float(s.replace(",", ""))\n\n\ndef _date(t: str):\n    t = t.lower()\n    m = re.search(r"\\b(\\d{4})-(\\d{2})-(\\d{2})\\b", t)\n    if m:\n        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"\n    m = re.search(r"\\b(\\d{1,2})[-/. ]?(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*[-/. ]?(\\d{2,4})\\b", t)\n    if m:\n        y = int(m.group(3)); y = y + 2000 if y < 100 else y\n        return f"{y:04d}-{MONTHS[m.group(2)]:02d}-{int(m.group(1)):02d}"\n    m = re.search(r"\\b(\\d{1,2})[-/.](\\d{1,2})[-/.](\\d{2,4})\\b", t)\n    if m:\n        y = int(m.group(3)); y = y + 2000 if y < 100 else y\n        return f"{y:04d}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"\n    return None\n\n\ndef parse(sms: str) -> dict:\n    t = sms.translate(_DEV)\n    low = t.lower()\n    out = {k: None for k in ["is_transaction", "type", "amount", "currency", "account_last4",\n                             "instrument", "counterparty", "date", "reference", "balance", "status"]}\n    if any(h in low for h in NEG_HINTS) and not any(h in low for h in ["debited from", "credited"]):\n        out["is_transaction"] = False\n        return out\n    out["is_transaction"] = True\n    is_credit = any(h in low for h in CREDIT_HINTS)\n    out["type"] = "credit" if is_credit else "debit"\n    amts = list(AMT_RE.finditer(t))\n    if amts:\n        m = amts[0]\n        out["amount"] = _num(m.group(1) or m.group(2))\n        head = m.group(0).lower()\n        out["currency"] = ("USD" if ("usd" in head or "$" in head) else "EUR" if ("eur" in head or "€" in head)\n                           else "GBP" if ("gbp" in head or "£" in head) else "AED" if "aed" in head else "INR")\n    m = re.search(r"(?:bal(?:ance)?|शेष(?: राशि)?)\\s*:?\\s*(?:is\\s*)?(?:rs\\.?|inr|₹|रु\\.?|रुपये)?\\s*([\\d,]+(?:\\.\\d+)?)", t, re.I)\n    if m:\n        out["balance"] = _num(m.group(1))\n    m = re.search(r"(?:x+|\\*+|ending\\s|-)(\\d{4})\\b", t, re.I)\n    if m:\n        out["account_last4"] = m.group(1)\n    for inst in ["NEFT", "IMPS", "RTGS", "NACH", "UPI", "ATM"]:\n        if inst.lower() in low:\n            out["instrument"] = inst\n            break\n    if "autopay" in low or "ऑटोपे" in low:\n        out["instrument"] = "AUTOPAY"\n    if "card" in low or "कार्ड" in low:\n        out["instrument"] = "CARD"\n    m = re.search(r"(?:ref(?:erence)?\\.?\\s*(?:no\\.?)?|utr|संदर्भ(?: संख्या)?|रेफ़)\\s*:?\\s*([A-Z0-9]{8,})", t, re.I)\n    if m:\n        out["reference"] = m.group(1)\n    m = re.search(r"(?:to|at|from|by|towards)\\s+(?:vpa\\s+)?([A-Za-z0-9.@ ]+?)(?=\\s+(?:on|via|for|ke|ko|se|par)\\b|[.,;(]|$)", t)\n    if m:\n        out["counterparty"] = m.group(1).strip()\n    out["date"] = _date(t)\n    out["status"] = ("failed" if any(h in low for h in ["fail", "declined", "विफल", "अस्वीकृत", "could not", "nahi ho", "नहीं हो सका", "decline"])\n                     else "reversed" if any(h in low for h in ["reversed", "वापस कर दिए", "wapas aa gaye"])\n                     else "success")\n    return out\n\n\ndef predict(sms: str) -> str:\n    return json.dumps(parse(sms), ensure_ascii=False)\n'
open(f"{WORK}/src/regex_baseline.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/regex_baseline.py', len(CONTENT), 'chars')

In [ ]:
sys.path.insert(0, f"{WORK}/src")
import importlib, smsgen, scoring as S, stage2_data as D
for m in (smsgen, S, D): importlib.reload(m)
from smsgen import gold_to_str, build_messages, SYSTEM_PROMPT

## 3. Data and the few-shot examples
The few-shot examples are picked with exactly the same code and seed as Tiny Aya's few-shot baseline in `stage2_train_eval.ipynb`.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi, login
if os.environ.get("HF_TOKEN"): login(token=os.environ["HF_TOKEN"])
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
files = {"train": "train_v3", **{s: s for s in SPLITS}}
splits = {k: load_jsonl(hf_hub_download(V3_DATA_REPO, f"data/{v}.jsonl", repo_type="dataset")) for k, v in files.items()}
if N_PER_SPLIT:
    splits = {k: (v if k == "train" else v[:N_PER_SPLIT]) for k, v in splits.items()}
for r in splits["unrelated"]: r.setdefault("tags", ["unrelated"])
train_rows = splits["train"]
_rng = random.Random(SEED)
def _pick(lang, txn):
    pool = [r for r in train_rows if r["lang"] == lang and r["gold"]["is_transaction"] == txn] \
        or [r for r in train_rows if r["gold"]["is_transaction"] == txn]
    clean = [r for r in pool if set(r["tags"]) <= {"hard_negative"}]
    return _rng.choice(clean or pool)
FEW_SHOT = [(r["sms"], gold_to_str(r["gold"])) for r in [_pick("en", True), _pick("hi", True), _pick("hinglish", False)]]
for s, g in FEW_SHOT: print("example:", s[:90], "->", g[:80])
print({k: len(v) for k, v in splits.items()})

## 4. Models (via CuratorKIT's `LiteLLMBackend`)

In [ ]:
import requests
from curatorkit.llm.litellm import LiteLLMBackend
def cohere_models():
    try:
        r = requests.get("https://api.cohere.com/v1/models", params={"endpoint": "chat", "page_size": 1000},
                         headers={"Authorization": f"Bearer {os.environ.get('COHERE_API_KEY', '')}"}, timeout=30)
        return [m["name"] for m in r.json().get("models", [])]
    except Exception as e:
        print("could not list Cohere models:", e); return []
names = cohere_models()
cmd = next((p for p in COMMAND_PREFERENCE if p in names), None) or next(
    (n for n in names if n.startswith("command-a") and not any(x in n for x in ["vision", "reasoning", "translate"])), COMMAND_PREFERENCE[0])
MODELS = {}
if RUN_COMMAND:
    MODELS["command_a"] = {"backend": LiteLLMBackend(model=f"cohere_chat/{cmd}", temperature=0.0, max_tokens=300, max_retries=6),
                           "rpm": COMMAND_RPM, "name": cmd}
if RUN_OPEN:
    MODELS["llama_3_3_70b"] = {"backend": LiteLLMBackend(model=f"openai/{OPEN_MODEL}", api_base="https://router.huggingface.co/v1",
                                                         api_key=os.environ.get("HF_TOKEN"), temperature=0.0, max_tokens=300, max_retries=4),
                               "rpm": OPEN_RPM, "name": OPEN_MODEL}
EXTRA = {"extra_headers": {"X-HF-Bill-To": HF_BILL_TO}} if HF_BILL_TO else {}
for k, m in MODELS.items():
    try:
        kw = EXTRA if k == "llama_3_3_70b" else {}
        out = m["backend"].generate([{"role": "user", "content": "Reply with the single word OK."}], max_tokens=5, **kw)
        print(k, m["name"], "-> probe OK:", out.text.strip()[:20])
    except Exception as e:
        print(k, m["name"], "-> probe FAILED:", str(e)[:300])
n_calls = sum(len(splits[s]) for s in SPLITS) * len(SETTINGS)
print(f"calls per model: {n_calls}")

## 5. Run (cached, rate-limited; JSON mode on)

In [ ]:
class RateLimiter:
    def __init__(self, rpm):
        self.dt, self.lock, self.next = 60.0 / max(rpm, 1), threading.Lock(), time.time()
    def wait(self):
        with self.lock:
            t = max(self.next, time.time()); self.next = t + self.dt
        time.sleep(max(0.0, t - time.time()))

def run_llm(task, items, make_messages, backend, rpm, workers=6, **gen_kwargs):
    path = f"{WORK}/cache/{task}.jsonl"
    cache = {}
    if os.path.exists(path):
        for rec in load_jsonl(path):
            if not rec.get("error"): cache[rec["id"]] = rec
    todo = [it for it in items if it["id"] not in cache]
    print(f"[{task}] cached {len(items) - len(todo)}, calling {len(todo)}")
    lim, lock, t0 = RateLimiter(rpm), threading.Lock(), time.time()
    fh = open(path, "a", encoding="utf-8")
    state = {"json_mode": True}
    def one(it):
        lim.wait()
        kw = dict(gen_kwargs)
        if state["json_mode"]: kw["response_format"] = {"type": "json_object"}
        t1 = time.time()
        try:
            r = backend.generate(make_messages(it), **kw)
            rec = {"id": it["id"], "text": r.text, "tokens_in": r.prompt_tokens, "tokens_out": r.completion_tokens,
                   "latency_s": round(time.time() - t1, 3), "json_mode": "response_format" in kw, "error": None}
        except Exception as e:
            msg = str(e)
            if "response_format" in msg or "json" in msg.lower() and "support" in msg.lower():
                state["json_mode"] = False
            rec = {"id": it["id"], "text": "", "error": f"{type(e).__name__}: {msg[:300]}"}
        with lock:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()
            if not rec["error"]: cache[it["id"]] = rec
        return rec
    errors = 0
    with cf.ThreadPoolExecutor(workers) as ex:
        for i, rec in enumerate(ex.map(one, todo), 1):
            errors += bool(rec["error"])
            if i % 100 == 0 or i == len(todo): print(f"  {i}/{len(todo)}, {errors} errors, {time.time() - t0:.0f}s")
    fh.close()
    if errors: print("  re-run this cell to retry only the failed calls")
    return cache

PROMPTS = {"zero_shot": lambda r: build_messages(r["sms"]), "few_shot": lambda r: build_messages(r["sms"], FEW_SHOT)}
RAW = {}
for mk, m in MODELS.items():
    for setting in SETTINGS:
        for sp in SPLITS:
            kw = EXTRA if mk == "llama_3_3_70b" else {}
            RAW[(mk, setting, sp)] = run_llm(f"{mk}_{setting}_{sp}", splits[sp], PROMPTS[setting], m["backend"], m["rpm"], **kw)

## 6. Score with the same scorer and AuditKIT

In [ ]:
ak = None
if OK_AUDIT:
    try:
        import auditkit as ak
    except Exception as e:
        print("AuditKIT import failed:", e)
RESULTS = {}
def pct(x): return "–" if x is None or (isinstance(x, float) and x != x) else f"{100 * x:.1f}"
for (mk, setting, sp), cache in RAW.items():
    rows = [r for r in splits[sp] if r["id"] in cache]
    outs = [cache[r["id"]]["text"] for r in rows]
    scored = [S.score_example(o, r["gold"]) for o, r in zip(outs, rows)]
    audit = {"auditkit": False}
    if ak is not None and rows:
        try:
            by_sms = {r["sms"]: o for r, o in zip(rows, outs)}
            res = ak.evaluate([ak.Sample(input=r["sms"], target=gold_to_str(r["gold"])) for r in rows],
                              model=lambda prompts: [by_sms.get(p, "") for p in prompts])
            audit = {"auditkit": True, "headline": str(getattr(res, "headline", ""))[:500]}
        except Exception as e:
            audit["auditkit_error"] = str(e)[:300]
    name = f"{mk}_{setting}"
    RESULTS[(name, sp)] = {"rows": rows, "outputs": outs, "scored": scored, "metrics": S.aggregate(scored),
                           "missing": len(splits[sp]) - len(rows), "audit": audit,
                           "tokens_in": [cache[r["id"]].get("tokens_in", 0) for r in rows],
                           "tokens_out": [cache[r["id"]].get("tokens_out", 0) for r in rows],
                           "latency": [cache[r["id"]].get("latency_s") for r in rows]}
    write_jsonl(f"{WORK}/results/predictions_{name}_{sp}.jsonl",
                [{"id": r["id"], "sms": r["sms"], "gold": r["gold"], "output": o, "exact_match": s["exact_match"]}
                 for r, o, s in zip(rows, outs, scored)])

KEYS = [("exact_match", "Exact match"), ("field_acc_txn", "Field acc."), ("hallucination_rate", "Halluc. ↓"),
        ("omission_rate", "Omission ↓"), ("nontxn_f1", "Non-txn F1"), ("schema_valid", "Schema valid")]
lines = []
for sp in SPLITS:
    lines += [f"\n### {sp}", "| System | n | EM [95% CI] | " + " | ".join(n for k, n in KEYS[1:]) + " |", "|---|---|---|" + "---|" * (len(KEYS) - 1)]
    for (name, s2), v in RESULTS.items():
        if s2 != sp: continue
        ci = S.bootstrap_ci([float(x["exact_match"]) for x in v["scored"]], seed=SEED)
        m = v["metrics"]
        lines.append(f"| {name} | {len(v['rows'])} | {pct(m.get('exact_match'))} [{pct(ci[0])}–{pct(ci[1])}] | "
                     + " | ".join(pct(m.get(k)) for k, _ in KEYS[1:]) + " |")
FRONTIER_TABLE = "\n".join(lines)
print(FRONTIER_TABLE)

## 7. Cost and latency

In [ ]:
cost_lines = ["| System | Avg input tokens | Avg output tokens | USD per 1M SMS (list price) | Median latency (s) |", "|---|---|---|---|---|"]
COST = {}
for mk in MODELS:
    for setting in SETTINGS:
        name = f"{mk}_{setting}"
        tin = [t for (n, sp), v in RESULTS.items() if n == name for t in v["tokens_in"] if t]
        tout = [t for (n, sp), v in RESULTS.items() if n == name for t in v["tokens_out"] if t]
        lat = sorted(l for (n, sp), v in RESULTS.items() if n == name for l in v["latency"] if l is not None)
        if not tin: continue
        ai, ao = sum(tin) / len(tin), sum(tout) / max(len(tout), 1)
        pin, pout = PRICES.get(mk, (float("nan"), float("nan")))
        usd = (ai * pin + ao * pout)   # per 1M SMS: tokens per SMS x USD per 1M tokens
        COST[name] = {"avg_in": ai, "avg_out": ao, "usd_per_1M_sms": usd, "median_latency_s": lat[len(lat) // 2] if lat else None}
        cost_lines.append(f"| {name} | {ai:.0f} | {ao:.0f} | {usd:,.2f} | {COST[name]['median_latency_s']} |")
COST_TABLE = "\n".join(cost_lines)
print(COST_TABLE)
print("Latency is wall-clock per API call from Kaggle with several calls in flight; treat it as indicative.")

## 8. Paired comparison with the fine-tuned Tiny Aya (same message ids)
Needs the predictions from `stage2_train_eval.ipynb`. If that run is not finished yet, run this cell again later.

In [ ]:
PAIR_TABLE = ""
try:
    lines = ["| Tiny Aya system | Frontier system | Set | Tiny Aya EM | Frontier EM | Tiny Aya-only correct | Frontier-only correct | McNemar p |",
             "|---|---|---|---|---|---|---|---|"]
    for ta in ["fine_tuned", "fire_v2"]:
        for sp in SPLITS:
            p = hf_hub_download(TINY_AYA_RESULTS_REPO, f"results/predictions_{ta}_{sp}.jsonl", repo_type="dataset")
            ta_em = {r["id"]: bool(r["exact_match"]) for r in load_jsonl(p)}
            for (name, s2), v in RESULTS.items():
                if s2 != sp: continue
                fr = {r["id"]: bool(x["exact_match"]) for r, x in zip(v["rows"], v["scored"])}
                ids = [i for i in fr if i in ta_em]
                if not ids: continue
                mc = S.mcnemar_exact([fr[i] for i in ids], [ta_em[i] for i in ids])
                lines.append(f"| {ta} | {name} | {sp} | {pct(sum(ta_em[i] for i in ids) / len(ids))} | {pct(sum(fr[i] for i in ids) / len(ids))} | "
                             f"{mc['b_only']} | {mc['a_only']} | {mc['p_value']:.2g} |")
    PAIR_TABLE = "\n".join(lines)
    print(PAIR_TABLE)
except Exception as e:
    print("Tiny Aya predictions not available yet:", str(e)[:200])

## 9. Save and upload (into the v3 dataset repo, `results/frontier/`)

In [ ]:
summary = {"models": {k: m["name"] for k, m in MODELS.items()}, "settings": SETTINGS, "prices_usd_per_1M_tokens": PRICES,
           "metrics": {f"{n}/{sp}": {**v["metrics"], "missing": v["missing"], "audit": v["audit"]} for (n, sp), v in RESULTS.items()},
           "cost": COST}
json.dump(summary, open(f"{WORK}/results/frontier_summary.json", "w"), indent=1, default=str)
open(f"{WORK}/results/frontier_tables.md", "w", encoding="utf-8").write(
    "## Frontier LLMs\n" + FRONTIER_TABLE + "\n\n## Cost and latency\n" + COST_TABLE + "\n\n## Paired with Tiny Aya\n" + PAIR_TABLE)
if not QUICK_MODE:
    HfApi().upload_folder(folder_path=f"{WORK}/results", path_in_repo="results/frontier", repo_id=V3_DATA_REPO,
                          repo_type="dataset", commit_message="Frontier LLM baselines (Command A, Llama 3.3 70B)")
    print("uploaded to", V3_DATA_REPO, "results/frontier/")
else:
    print("QUICK_MODE: not uploaded")